# Preprocessing Pipelines from Scratch

Scalers, imputer, encoders, `ColumnTransformer` and `Pipeline` in numpy. The key rule: **fit on training data only**, then apply the learned statistics to test data.

In [ ]:
import numpy as np

class StandardScaler:
    def fit(self, X):
        X = np.asarray(X, float)
        self.mean_ = np.nanmean(X, axis=0)
        self.scale_ = np.nanstd(X, axis=0)
        self.scale_[self.scale_ == 0] = 1.0   # constant column: leave centred, avoid /0
        return self
    def transform(self, X):
        return (np.asarray(X, float) - self.mean_) / self.scale_
    def inverse_transform(self, Z):
        return np.asarray(Z) * self.scale_ + self.mean_

class MinMaxScaler:
    def __init__(self, feature_range=(0, 1)):
        self.feature_range = feature_range
    def fit(self, X):
        X = np.asarray(X, float)
        self.min_, self.max_ = np.nanmin(X, axis=0), np.nanmax(X, axis=0)
        self.range_ = np.where(self.max_ - self.min_ == 0, 1.0, self.max_ - self.min_)
        return self
    def transform(self, X):
        lo, hi = self.feature_range
        return (np.asarray(X, float) - self.min_) / self.range_ * (hi - lo) + lo

class SimpleImputer:
    def __init__(self, strategy='mean'):
        self.strategy = strategy
    def fit(self, X):
        X = np.asarray(X, float)
        f = {'mean': np.nanmean, 'median': np.nanmedian}[self.strategy]
        self.fill_ = f(X, axis=0)
        return self
    def transform(self, X):
        X = np.array(X, float)
        r, c = np.where(np.isnan(X))
        X[r, c] = self.fill_[c]
        return X

class OneHotEncoder:
    """Categories learned at fit time; unseen categories map to all zeros."""
    def fit(self, X):
        X = np.asarray(X, dtype=object).reshape(len(X), -1)
        self.categories_ = [sorted(set(X[:, j])) for j in range(X.shape[1])]
        return self
    def transform(self, X):
        X = np.asarray(X, dtype=object).reshape(len(X), -1)
        cols = []
        for j, cats in enumerate(self.categories_):
            idx = {c: i for i, c in enumerate(cats)}
            block = np.zeros((len(X), len(cats)))
            for i, v in enumerate(X[:, j]):
                if v in idx:
                    block[i, idx[v]] = 1
            cols.append(block)
        return np.hstack(cols)

class OrdinalEncoder:
    def __init__(self, order=None):
        self.order = order
    def fit(self, X):
        X = np.asarray(X, dtype=object).reshape(len(X), -1)
        self.categories_ = self.order or [sorted(set(X[:, j])) for j in range(X.shape[1])]
        return self
    def transform(self, X):
        X = np.asarray(X, dtype=object).reshape(len(X), -1)
        out = np.full(X.shape, np.nan)
        for j, cats in enumerate(self.categories_):
            idx = {c: i for i, c in enumerate(cats)}
            out[:, j] = [idx.get(v, np.nan) for v in X[:, j]]
        return out

class ColumnTransformer:
    def __init__(self, transformers):
        self.transformers = transformers   # [(name, transformer, [col indices])]
    def fit(self, X):
        X = np.asarray(X, dtype=object)
        for _, t, cols in self.transformers:
            t.fit(X[:, cols])
        return self
    def transform(self, X):
        X = np.asarray(X, dtype=object)
        return np.hstack([np.asarray(t.transform(X[:, cols]), float) for _, t, cols in self.transformers])

class Pipeline:
    def __init__(self, steps):
        self.steps = steps   # [(name, obj)]; last may be a model
    def fit(self, X, y=None):
        for _, s in self.steps[:-1]:
            X = s.fit(X).transform(X)
        last = self.steps[-1][1]
        last.fit(X, y) if y is not None else last.fit(X)
        return self
    def _pre(self, X):
        for _, s in self.steps[:-1]:
            X = s.transform(X)
        return X
    def transform(self, X):
        return self.steps[-1][1].transform(self._pre(X))
    def predict(self, X):
        return self.steps[-1][1].predict(self._pre(X))


## Check against sklearn

In [ ]:
from sklearn.preprocessing import StandardScaler as SkS, MinMaxScaler as SkM, OneHotEncoder as SkO
from sklearn.impute import SimpleImputer as SkI
Xn = np.random.RandomState(0).randn(50, 3) * 5 + 2
print(np.allclose(StandardScaler().fit(Xn).transform(Xn), SkS().fit_transform(Xn)))
print(np.allclose(MinMaxScaler().fit(Xn).transform(Xn), SkM().fit_transform(Xn)))
Xm = Xn.copy(); Xm[::7, 1] = np.nan
print(np.allclose(SimpleImputer('median').fit(Xm).transform(Xm), SkI(strategy='median').fit_transform(Xm)))
c = np.array([['a'], ['b'], ['c'], ['a']], dtype=object)
print(np.allclose(OneHotEncoder().fit(c).transform(c), SkO().fit_transform(c).toarray()))

## Leakage and unseen categories
Statistics come from the training split; an unseen category becomes an all-zero row instead of crashing.

In [ ]:
tr, te = np.array([[1.], [2.], [3.]]), np.array([[100.]])
print(StandardScaler().fit(tr).transform(te))      # uses train mean/std, not test
print(OneHotEncoder().fit([['a'], ['b']]).transform([['z']]))

## End-to-end: mixed numeric/categorical data with missing values

In [ ]:
rng = np.random.RandomState(0)
n = 400
age = rng.normal(40, 12, n); income = np.exp(rng.normal(10.5, 0.5, n))
city = rng.choice(['NYC', 'LA', 'SF'], n); edu = rng.choice(['hs', 'bsc', 'msc'], n, p=[.4,.4,.2])
y = ((age > 40) ^ (city == 'SF') | (edu == 'msc')).astype(int)
age[rng.rand(n) < .1] = np.nan; income[rng.rand(n) < .1] = np.nan
X = np.empty((n, 4), dtype=object); X[:, 0], X[:, 1], X[:, 2], X[:, 3] = age, income, city, edu
perm = rng.permutation(n); tr, te = perm[:300], perm[300:]

class LogReg:
    def __init__(self, lr=0.1, epochs=500): self.lr, self.epochs = lr, epochs
    def fit(self, X, y):
        self.w, self.b = np.zeros(X.shape[1]), 0.0
        for _ in range(self.epochs):
            p = 1 / (1 + np.exp(-(X @ self.w + self.b)))
            self.w -= self.lr * X.T @ (p - y) / len(y); self.b -= self.lr * (p - y).mean()
        return self
    def predict(self, X): return (X @ self.w + self.b > 0).astype(int)

pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer('median')), ('sc', StandardScaler())]), [0, 1]),
    ('cat', OneHotEncoder(), [2]),
    ('ord', OrdinalEncoder(order=[['hs', 'bsc', 'msc']]), [3]),
])
model = Pipeline([('pre', pre), ('clf', LogReg())]).fit(X[tr], y[tr])
print('pipeline test acc:', (model.predict(X[te]) == y[te]).mean())
